# 02 - Modeling

Train models for credit card fraud detection and evaluate them with precision-recall metrics.

## Setup

Add the project root to the path so we can import from `src`.

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

from sklearn.metrics import average_precision_score

from src.data import load_processed, split_data
from src.evaluate import metrics_at
from src.features import add_features
from src.train import train_baseline

## Data

Load the processed data and make the stratified train/test split. The engineered features are added after the split, separately on each set.

In [ ]:
X_train, X_test, y_train, y_test = split_data(load_processed())
X_train = add_features(X_train)
X_test = add_features(X_test)

print("Train:", X_train.shape, "Test:", X_test.shape)

## Baseline: Logistic Regression

`StandardScaler` + `LogisticRegression(class_weight="balanced")`. This is a simple reference point to compare against the Random Forest later.

We report PR-AUC (average precision) and precision, recall and F1 at the default threshold of 0.5. The threshold is fixed, not tuned on the test set.

In [ ]:
baseline = train_baseline(X_train, y_train)
baseline_proba = baseline.predict_proba(X_test)[:, 1]

baseline_metrics = metrics_at(y_test, baseline_proba, threshold=0.5)
baseline_metrics["pr_auc"] = average_precision_score(y_test, baseline_proba)

baseline_metrics

## Random Forest: comparing `class_weight`

We compare three `class_weight` settings with stratified 3-fold cross-validation **on the training set only** (the test set is not touched). Each model uses 100 trees to keep it fast. The score is PR-AUC (average precision).

In [ ]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_score

from src.config import SEED
from src.train import make_rf

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)

rows = []
for class_weight in [None, "balanced", "balanced_subsample"]:
    model = make_rf(n_estimators=100, class_weight=class_weight)
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="average_precision")
    rows.append({"class_weight": str(class_weight), "pr_auc_mean": scores.mean(), "pr_auc_std": scores.std()})

cv_results = pd.DataFrame(rows).round(4)
cv_results

### Result

All three settings reach a very similar mean PR-AUC (about 0.835-0.838). `None` is slightly higher (0.8377), but the differences between settings (about 0.002) are much smaller than the variation across folds (std between 0.005 and 0.014). So, with 3 folds and 100 trees, **this comparison does not show a meaningful difference** between the `class_weight` options.

This makes sense: PR-AUC is based on the ranking of the predicted probabilities, and class weights mainly shift the probabilities rather than changing that ranking much. The choice of `class_weight` can still matter for the decision threshold, which we will set separately.

All Random Forest settings score well above the Logistic Regression baseline on PR-AUC (0.713), but note that the baseline was measured on the test set while these are cross-validation scores on the training set, so the numbers are only roughly comparable.